In [ ]:
import subprocess, sys, numpy, pandas
from importlib.metadata import version
from packaging.version import Version

pins = [f"numpy=={numpy.__version__}", f"pandas=={pandas.__version__}"]

def install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages, *pins],
                   check=True)

install("m3-sc")
if Version(version("scanpy")) < Version("1.12.5"):
    install("pynndescent<0.7")

# Use your own data

The other tutorials start from the built-in demo. This one starts from files on
disk, in the layout most multimodal datasets are shared in: **one `.h5ad` per
modality** (here RNA and ADT) for the same cells. It ends with a trained model and
the embedding stored back in your AnnData.

If your files are laid out differently (one set of files per batch, or one file
holding every modality), section 6 shows what to change. The tutorial runs in
about 3 minutes on a laptop CPU.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import m3

sc.settings.verbosity = 1

## 0. Example files

So that there is something to read, this cell writes 3,000 cells of the demo to
`rna.h5ad` and `adt.h5ad`. **Skip it when you have your own files.**

In [ ]:
demo = m3.datasets.liu_demo(n_cells=3000)
obs = demo.obs.copy()
obs.index = [f"cell{i:04d}" for i in range(demo.n_cells)]          # cell barcodes
order = np.random.default_rng(0).permutation(demo.n_cells)         # ADT file: other cell order

ad.AnnData(demo.modalities["rna"], obs=obs,
           var=pd.DataFrame(index=demo.var["rna"])).write_h5ad("rna.h5ad")
ad.AnnData(demo.modalities["adt"][order], obs=obs[["batch"]].iloc[order],
           var=pd.DataFrame(index=demo.var["adt"])).write_h5ad("adt.h5ad")

## 1. What M3 needs

| | |
|---|---|
| **Counts** | Raw, un-normalised counts for every modality. M3 normalises internally. |
| **Modality names** | Exactly `"rna"`, `"adt"` and/or `"atac"`. |
| **Cells** | Within a batch, the same cells in every modality, in the same order. |
| **Features** | Unique names per modality, named the same way in every batch. |
| **`obs` columns** | A batch column (for example the sequencing run) and one or two condition columns (for example disease status). The donor-level tasks also need a cell-type column and a donor column. |

## 2. Read the files and match the cells

The two files hold the same cells but not necessarily in the same order, so keep
the shared barcodes and put both objects in one order. Cells found in only one
file are dropped.

In [ ]:
rna = ad.read_h5ad("rna.h5ad")
adt = ad.read_h5ad("adt.h5ad")
print(rna)
print(adt)

cells = rna.obs_names.intersection(adt.obs_names)
rna = rna[cells].copy()
adt = adt[cells].copy()
print("shared cells:", len(cells))

Raw counts are whole numbers; normalised values have decimals. Look at a corner
of each matrix:

In [ ]:
print(rna.X[:5, :5].toarray())
print(adt.X[:5, :5].toarray())

If `.X` is normalised and the counts are in a layer (`print(rna)` then lists a
`counts` layer), pass the layer in the next step: `rna.layers["counts"]` in
place of `rna.X`. Given normalised values, M3 warns `modality 'rna' has
non-integer values; m3 expects RAW counts` but does not stop; no such warning in
step 3 means every matrix passed that check.

## 3. Build the Dataset

`m3.read_matrix` takes one count matrix per modality, the per-cell table and the
feature names. Add the barcodes as a column of `obs`: the model returns its
results in its own row order, and this column is what matches them back to your
cells in step 5.

In [ ]:
obs = rna.obs.copy()
obs["obs_names"] = rna.obs_names

data = m3.read_matrix(
    counts={"rna": rna.X, "adt": adt.X},
    obs=obs,
    var={"rna": rna.var_names, "adt": adt.var_names},
)
print(data)

The three batches listed here come from the `obs` column called `batch`. If your
batch column has another name, leave it as it is: `print(data)` then shows
`batches=not set`, and you name the column in the next step.

## 4. Train

Tell the model which `obs` columns to use:

- `condition_keys` — the condition column(s), at most two. Required.
- `batch_key` — the batch column, when it is not called `batch`, for example
  `batch_key="run"`. If the model finds no batch column it stops with an error
  that lists the columns of `obs`.

`model.contract["batches"]` lists the batches the model will use.

With these two, `train()` fits the cell-level model, which is all the embedding
needs. The lines printed before the epochs are internal messages;
`Batch counts` is the number of cells per batch, in the order of
`model.contract["batches"]`.

In [ ]:
model = m3.M3(data, condition_keys="cond_group")
print("batches:", model.contract["batches"])

model.train()
print("capabilities:", model.capabilities)

## 5. Put the embedding back in your AnnData

Index the embedding by the `obs_names` column of `model.cell_metadata`, then
reorder it to your AnnData (which now holds the shared cells only). `part="bio"` includes the condition factors, so cells
also separate by condition; `part="intrinsic"` leaves them out.

In [ ]:
emb = pd.DataFrame(model.embedding(part="bio"),
                   index=model.cell_metadata["obs_names"])
rna.obsm["X_m3"] = emb.loc[rna.obs_names].to_numpy()
print(rna.obsm["X_m3"].shape)

# check the matching: the model's metadata, reordered the same way, equals yours
meta = model.cell_metadata.set_index("obs_names").loc[rna.obs_names]
assert (meta["mergedcelltype"].astype(str).to_numpy() == rna.obs["mergedcelltype"].astype(str).to_numpy()).all()

In [ ]:
sc.pp.neighbors(rna, use_rep="X_m3")
sc.tl.umap(rna)
sc.pl.umap(rna, color=["mergedcelltype", "batch", "cond_group"], wspace=0.5, show=False)
plt.savefig("umap_m3.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
rna.write_h5ad("rna_m3.h5ad")          # your AnnData, with the embedding in obsm["X_m3"]

From here everything works as in the other tutorials. Add `celltype_key="..."`
and `donor_key="..."` to `m3.M3` for
[patient prediction](https://pyanglab.github.io/M3/notebooks/py_02_patient_prediction/) and
[attribution](https://pyanglab.github.io/M3/notebooks/py_03_attribution/).

## 6. Other file layouts

Only steps 2 and 3 change. Steps 4 and 5 stay as they are. The cells below make
their example files from the objects above, so with your own data take only the
part that matches your layout.

### One set of files per batch

Build one Dataset per batch, then stack them with `m3.concat`. Files like these
do not say which batch they are, so name it with `batch=`. Features are matched by name and only those present in every
batch are kept. Cell names must be unique across batches for step 5; prefix them
with the batch name if they are not. The order in which a Dataset lists its
modalities does not matter.

In [ ]:
for b in ["B1", "B2", "B3"]:                       # example files, one pair per batch
    keep = rna.obs_names[rna.obs["batch"] == b]
    part = rna[keep, :].copy()
    del part.obs["batch"]                          # the batch is only in the file name
    part.write_h5ad(f"{b}_rna.h5ad")
    adt[keep, :].write_h5ad(f"{b}_adt.h5ad")

In [ ]:
def load_batch(batch, rna_path, adt_path):
    rna = ad.read_h5ad(rna_path)
    adt = ad.read_h5ad(adt_path)
    cells = rna.obs_names.intersection(adt.obs_names)
    rna, adt = rna[cells], adt[cells]
    obs = rna.obs.copy()
    obs["obs_names"] = rna.obs_names
    return m3.read_matrix(
        counts={"rna": rna.X, "adt": adt.X},
        obs=obs,
        var={"rna": rna.var_names, "adt": adt.var_names},
        batch=batch,
    )

data = m3.concat([load_batch(b, f"{b}_rna.h5ad", f"{b}_adt.h5ad")
                  for b in ["B1", "B2", "B3"]])
print(data)

If the `obs` of every file already has a batch column, leave `batch=` out; a
label that differs from that column is an error. For batches that do not all have the same modalities, see
[Mosaic integration](https://pyanglab.github.io/M3/notebooks/py_06_mosaic_integration/).

### One `.h5ad` holding every modality

If all features sit in one AnnData and `var["feature_types"]` says which is
which (`Gene Expression`, `Antibody Capture`, `Peaks`, as written by Cell
Ranger), `m3.from_anndata` replaces steps 2 and 3. It splits the modalities and
adds the `obs_names` column itself. The column must be called `feature_types` and
hold exactly these values; features with any other value are left out, so check
the modalities in `print(data)`. Name the batch column with `batch_key=` in
step 4 if it is not called `batch`, and use this AnnData in place of `rna` in
step 5.

In [ ]:
protein = adt.copy()                                # example file with both modalities
protein.var_names = protein.var_names + "_protein"
both = ad.concat([rna, protein], axis=1, merge="first")
both.var["feature_types"] = ["Gene Expression"] * rna.n_vars + ["Antibody Capture"] * adt.n_vars
both.write_h5ad("multimodal.h5ad")

In [ ]:
both = ad.read_h5ad("multimodal.h5ad")
data = m3.from_anndata(both)
print(data)

If some batches in that file lack a modality, split the AnnData by batch, drop
the features of the missing modality, and stack the parts:

In [ ]:
parts = []
for b in ["B1", "B2", "B3"]:
    part = both[both.obs["batch"] == b]
    if b == "B3":                                          # as if B3 had no ADT
        part = part[:, part.var["feature_types"] == "Gene Expression"]
    parts.append(m3.from_anndata(part))                    # `obs` has the batch column

data = m3.concat(parts)
print(data)
print("cells measured:", {name: int(has.sum()) for name, has in data.present.items()})

### Seurat or SingleCellExperiment

Use the R package: see
[Use your own data (R)](https://pyanglab.github.io/M3/notebooks/r_05_own_data/).

## 7. Check before a long run

- **Modality names.** A modality with any other name than `rna`, `adt`, `atac`
  is left out of the model without an error.
- **Empty cells.** A cell whose counts in one modality are all zero is treated
  as not measured in that modality during training, whatever `data.present` says.
- **Number of features.** Counts are converted to dense matrices for training,
  so keep the variable features rather than every gene or peak. The cell below
  keeps the top genes and leaves the counts raw.
- **Batch sizes.** With the default `balance_batches=True`, training uses the
  same number of cells from every batch, set by the smallest one. The embedding
  is still returned for all cells.

In [ ]:
tmp = rna.copy()
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)
sc.pp.highly_variable_genes(tmp, n_top_genes=500, batch_key="batch")
rna_hvg = rna[:, tmp.var["highly_variable"]].copy()       # still raw counts
print(rna_hvg.shape)